# Supplementary Figure S5 — comparative behavioural data

Run cells in order. Calculated panels are saved under `figures/`.

In [ ]:
from pathlib import Path
import sys

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "notebooks").is_dir() and (p / "data").is_dir())
sys.path.insert(0, str(ROOT / "src"))
FIGURES = ROOT / "figures"
FIGURES.mkdir(parents=True, exist_ok=True)
SAVED_RESULTS = ROOT / "saved_results"

%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np
plt.style.use(ROOT / "notebooks/ncb.mplstyle")
plt.close("all")
np.random.seed(0)


## Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.cluster.hierarchy import linkage, dendrogram, fcluster
from scipy.spatial.distance import pdist
import numpy as np
from matplotlib.gridspec import GridSpec

cmp = "Greys"

MAX = 40
step = 10
cpauMax = 1000 + MAX
max_len = cpauMax

## Load the experimental dataset and fitted results

In [ ]:
import csv
d = np.load(SAVED_RESULTS / 'results_data.npz', allow_pickle=True)
cpfs_x = d['cpfs_x'].tolist()
cpfs_y = d['cpfs_y'].tolist()
cpfs_z = d['cpfs_z'].tolist()
archive = np.load(SAVED_RESULTS / 'fits_best_available.npz', allow_pickle=True)
cdc_x, cper_x = archive['CDC_X'], archive['CPER_X']
targets = archive['targets'].tolist()
fits = archive['fits']
fit_species = np.array([fit['species'] for fit in fits])
with (ROOT / 'data/species_names.csv').open(newline='') as source:
    aliases = {row['species_code']: row['figure_code'] for row in csv.DictReader(source)}
species_labels = np.array([aliases.get(code, code) for code in fit_species])
from species_profiles import targets_from_data
data_targets = targets_from_data(d, cdc_x, cper_x)

## Extract species curves and parameters

In [ ]:
cdc_Y = []
cper_Y = []
cdc_Ym = []
cper_Ym = []
cdc_Ys = []
cper_Ys = []
losses = []
inh_mults = []
ada_mults = []
for fit in fits:
    cdc_target, cper_target = targets[fit['species']]
    cdc_Y.append(cdc_target)
    cper_Y.append(cper_target)
    cdc_Ym.append(fit['cdc'])
    cper_Ym.append(fit['cper'])
    cdc_Ys.append(fit['seed_cdc'])
    cper_Ys.append(fit['seed_cper'])
    losses.append(fit['loss'])
    inh_mults.append(fit['inh_mult'])
    ada_mults.append(fit['ada_mult'])
cdc_Y = np.array(cdc_Y)
cper_Y = np.array(cper_Y)
cdc_Ys = np.array(cdc_Ys)
cper_Ys = np.array(cper_Ys)

## Cluster the endpoint duty-cycle responses

In [ ]:
num_clusters = 3
Z = linkage(pdist(cdc_Y[:,-2:], metric="euclidean"), method="average", optimal_ordering=True,)
labels = fcluster(Z, t=num_clusters, criterion="maxclust") - 1
cut = (Z[-num_clusters, 2] + Z[-num_clusters + 1, 2]) / 2

## Plot experimental fields with their matching profiles (S5)

These profiles are resampled from `results_data.npz`, using the same function as the fitter. For the corrected refits, the assertion above verifies that they are identical to the targets embedded in the fit archive.

In [ ]:
plotted_species = fit_species

fig, axs = plt.subplots(len(plotted_species)//2, 3*2, figsize=(10, 10))
for cnt, species in enumerate(plotted_species):
    ax = axs[cnt % 7, 0 + (cnt % 2 * 3)]
    im = ax.contourf(cpfs_x[species]/1000, cpfs_y[species]/1000, cpfs_z[species] / np.nanmax(cpfs_z[species]), levels=10, cmap="Greys")
    ax.axis('square')
    xmax, ymax = np.max(cpfs_x[species])/1000, np.max(cpfs_y[species])/1000
    if xmax>1:
        ax.set_xlim(0, 2)
        ax.set_ylim(0, 2)
        ax.set_xticks([0, 2])
        ax.set_yticks([0, 2])
        ax.text(0.8, 1.9, species_labels[cnt], fontsize=10)
    else:
        ax.set_xlim(0, 1)
        ax.set_ylim(0, 1)
        ax.set_xticks([0, 1])
        ax.set_yticks([0, 1])
        ax.text(0.4, 0.95, species_labels[cnt], fontsize=10)

    ax = axs[cnt % 7, 1 + (cnt % 2 * 3)]
    ax.plot(cdc_x, data_targets[species][0], c='k', label='data')
    ax.set_xticks([0, 1 ], labels=[])
    ax.set_yticks([0, 1])

    ax = axs[cnt % 7, 2 + (cnt % 2 * 3)]
    ax.plot(cper_x / 1000, data_targets[species][1], c='k', label='data')
    ax.set_xlim(0, 1)
    ax.set_xticks([0, 1], labels=[])
    ax.set_yticks([0, 1], labels=[])

axs[cnt % 7, 0 + (cnt % 2 * 3)].set_xlabel('Chirp\nduration [s]')
axs[cnt % 7, 0 + (cnt % 2 * 3)].set_ylabel('Chirp pause [s]')
axs[cnt % 7, 0].set_xlabel('Chirp\nduration [s]')
axs[cnt % 7, 0].set_ylabel('Chirp pause [s]')

axs[-1, 1].set_xticks([0, 1], labels=[0,  1 ])
axs[-1, 1].set_xlabel('Chirp\nduty cycle')

axs[-1, 2].set_xticks([0, 1], labels=[0, 1])
axs[-1, 2].set_xlabel('Chirp\nperiod [s]')

axs[-1, 4].set_xticks([0,  1 ], labels=[0,  1 ])
axs[-1, 4].set_xlabel('Chirp\nduty cycle')

axs[-1, 5].set_xticks([0,  1], labels=[0,  1])
axs[-1, 5].set_xlabel('Chirp\nperiod [s]')

fig.savefig(f'{FIGURES}/fig_S5.pdf', bbox_inches='tight')